In [ ]:
import pathlib
import os
import shutil
from collections import defaultdict
import datetime

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"

import numpy as np
import torch
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.tensorboard.writer import SummaryWriter
from pytorch3d.loss import chamfer_distance

import trimesh

import network, utils



%load_ext autoreload
%autoreload 2

/home/nikola/miniconda3/envs/adlr/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [ ]:
if torch.cuda.is_available():
    print("Using GPU:", torch.cuda.get_device_name(0))
    device = torch.device("cuda")
else:
    print("Using CPU")
    device = torch.device("cpu")

pcd = trimesh.load(pathlib.Path("data/pointcloud10.obj"), file_type = 'obj', force='pointcloud')
target_pc = torch.tensor(pcd.vertices, dtype=torch.float32).to(device)
target_pc = target_pc.unsqueeze(0) # Change [2048, 3] -> [1, 2048, 3]
# print(target_pc) # check

denoiser = network.Denoiser().to(device)
diffuser = network.Diffuser(timesteps=1000).to(device)
optimizer = Adam(denoiser.parameters(), lr=1e-5)


total, trainable = utils.count_parameters(denoiser)
print(f"Total: {total:,} | Trainable: {trainable:,} | Model size: {utils.model_memory_size(denoiser):.3f} MB")

Using GPU: NVIDIA GeForce MX130
Total: 692,739 | Trainable: 692,739 | Model size: 2.643 MB


In [ ]:
epochs = 5000 
batch_size = 32

# Model name
current_time = datetime.datetime.now().strftime("%b%d_%H-%M")
experiment_name = "Resnet_2Blocks_256Hidden_reweighted_loss"
denoiser_id = f"{current_time}_{experiment_name}_{epochs}_epochs"

# Create tensorboard writer    
log_path = pathlib.Path(f"logs/diffusion_training/{denoiser_id}")
writer = SummaryWriter(log_path)
# For tracking loss per timestep
timestep_loss_sum = defaultdict(float)
timestep_counts = defaultdict(int)
#Run this code in terminal to start tensorboard: tensorboard --logdir=diffusion/nikola/logs/diffusion_training

batch_target_pc = target_pc.repeat(batch_size, 1, 1)

for epoch in range(epochs):
    denoiser.train()
    optimizer.zero_grad()
    
    # Pick a random timestep for each item in the batch
    t = torch.randint(0, diffuser.t, (batch_size,), device=device).long()
    
    # Forward Process: Add noise to the clean shape
    noisy_pc, actual_noise = diffuser.add_noise(batch_target_pc, t)
    
    # Backward Process: Predict the noise
    predicted_noise = denoiser(noisy_pc, t)
    
    # --- MIN-SNR GAMMA WEIGHTING ---

    alpha_bar = diffuser.alpha_cumprod[t]
    snr = alpha_bar / (1 - alpha_bar)
    gamma = 5.0
    mse_loss_weights = torch.clamp(snr, max=gamma) / snr

    MSEloss = F.mse_loss(predicted_noise, actual_noise, reduction='none').mean(dim=[1, 2])
    MSESNRloss = (MSEloss * mse_loss_weights).mean()
    # --- MIN-SNR GAMMA WEIGHTING --- 

    # Chamfer Loss

    alpha_cumprod = diffuser.alpha_cumprod[t].view(-1, 1, 1)
    pred_x0 = (noisy_pc - torch.sqrt(1 - alpha_cumprod) * predicted_noise) / torch.sqrt(alpha_cumprod)
    Chamferloss, _ = chamfer_distance(pred_x0, target_pc)
    
    # Chamfer Loss

    MSESNRloss.backward()
    optimizer.step()

    writer.add_scalar("MSE Loss", MSEloss.item(), epoch)
    writer.add_scalar("MSESNR Loss", MSESNRloss.item(), epoch)
    writer.add_scalar("Chamfer Loss", Chamferloss.item(), epoch)
    # timestep_loss_sum[t.item()] += MSEloss.item()
    # timestep_counts[t.item()] += 1
    for ti, loss_val in zip(t.tolist(), MSEloss.tolist()):
        timestep_loss_sum[ti] += loss_val
        timestep_counts[ti] += 1
    print(f"Epoch {epoch}, T = {t.item()} | MSE Loss: {MSEloss.item():.6f}| Chamfer Loss: {Chamferloss.item():.6f}")

for time_val in range(diffuser.t):
        if timestep_counts[time_val] > 0:
            avg_error = timestep_loss_sum[time_val] / timestep_counts[time_val]
            writer.add_scalar('Timestep Error', avg_error, global_step=time_val)

utils.save_model(denoiser, diffuser, epoch, denoiser_id)


Epoch 0, T = 938 | MSE Loss: 0.991345| Chamfer Loss: 22097.339844
Epoch 1, T = 644 | MSE Loss: 1.251958| Chamfer Loss: 227.845871
Epoch 2, T = 910 | MSE Loss: 0.775297| Chamfer Loss: 10180.910156
Epoch 3, T = 106 | MSE Loss: 0.874253| Chamfer Loss: 0.085060
Epoch 4, T = 562 | MSE Loss: 0.420416| Chamfer Loss: 22.663317
Epoch 5, T = 212 | MSE Loss: 0.681207| Chamfer Loss: 0.251354
Epoch 6, T = 673 | MSE Loss: 0.226975| Chamfer Loss: 55.678360
Epoch 7, T = 410 | MSE Loss: 0.218169| Chamfer Loss: 1.066303
Epoch 8, T = 185 | MSE Loss: 0.454963| Chamfer Loss: 0.075709
Epoch 9, T = 536 | MSE Loss: 0.160167| Chamfer Loss: 5.179834
Epoch 10, T = 275 | MSE Loss: 0.314718| Chamfer Loss: 0.288675
Epoch 11, T = 752 | MSE Loss: 0.153466| Chamfer Loss: 145.761826
Epoch 12, T = 911 | MSE Loss: 0.082971| Chamfer Loss: 1075.682251
Epoch 13, T = 549 | MSE Loss: 0.191651| Chamfer Loss: 6.793698
Epoch 14, T = 399 | MSE Loss: 0.168966| Chamfer Loss: 0.337057
Epoch 15, T = 241 | MSE Loss: 0.268801| Chamfer 

In [17]:
#Load a model:
denoiser_id = "May13_14-53_Resnet_2Blocks_256Hidden_reweighted_loss_10000_epochs"
utils.reload_model(denoiser, diffuser, denoiser_id, device)

RuntimeError: Error(s) in loading state_dict for Denoiser:
	Unexpected key(s) in state_dict: "layer1.weight", "layer1.bias". 

In [20]:
generateDDIM = True
generateDDPM = False
number_of_points = 2056
DDIM_steps = 200
number_of_DDIM_iterations = 1

if generateDDIM:
    for _ in range(number_of_DDIM_iterations):
        generated_pc_ddim = network.sample_ddim(denoiser, diffuser, n_points=number_of_points, steps=DDIM_steps)
        generated_pcd_ddim = trimesh.PointCloud(generated_pc_ddim.squeeze().cpu().numpy())
        utils.visualize_comparison(target_pc, generated_pc_ddim, window_name="DDIM Target (Red) vs Generated (Blue)")

if generateDDPM:
    generated_pc_ddpm = network.sample_ddpm(denoiser, diffuser, n_points=number_of_points)
    generated_pcd_ddpm = trimesh.PointCloud(generated_pc_ddpm.squeeze().cpu().numpy())
    utils.visualize_comparison(target_pc, generated_pc_ddpm, window_name="DDPM Target (Red) vs Generated (Blue)")




Visualizing: Target is RED, Generated is BLUE.


In [ ]:
generated_pc, samples_list = network.sample_and_capture(denoiser, diffuser, n_points=number_of_points, save_every=10)
utils.visualize_diffusion_progress(samples_list, window_name="Diffusion Process")

In [35]:

# Optionally, save the generated point clouds to disk
generated_pcd_ddim.export(f"output/{denoiser_id}_ddim.obj")
generated_pcd_ddpm.export(f"output/{denoiser_id}_ddpm.obj")

'# https://github.com/mikedh/trimesh\nv -0.56441766 0.78872430 -0.18467656\nv 0.69580579 -0.20013197 0.61616433\nv -0.86288488 0.49203083 -0.15962587\nv 0.48783290 0.35668495 0.40350926\nv 0.98029727 0.09194494 -0.21980199\nv 0.02946977 -0.98523468 -0.26776981\nv 0.00129082 0.86175340 0.01463670\nv 0.53662318 -0.27286988 0.50937229\nv -0.26935858 0.32102132 -0.18292204\nv -0.33940518 0.38525423 -0.46772867\nv -0.69208896 -0.21610272 0.18053158\nv -0.68740106 -0.43805766 -0.26573423\nv -0.52584171 -0.30063847 0.30289534\nv 0.05930847 -0.19985540 -0.29118839\nv -0.02350364 0.86983013 -0.38706601\nv 0.81569612 0.53392655 -0.34571153\nv -1.06174374 -0.03352182 0.02351772\nv 0.07378376 -0.77049774 -0.10110997\nv 0.25538582 -0.28099233 0.58130467\nv -0.25627917 0.23873901 -0.42154562\nv -0.65982777 0.55066711 -0.30719045\nv -0.80864823 -0.60709095 0.05355192\nv -0.57824832 0.52951932 -0.42947111\nv 1.06332219 -0.11305494 -0.09279465\nv 0.27940497 0.43306753 -0.00727826\nv -0.79350215 -0.4030